# Create Silver Tables

For this basic analysis we want to split the buildings into residential and commercial types. 

In [ ]:
!pip install geopandas folium

In [ ]:
from pyspark.sql import functions as F

In [ ]:
# Set up widgets for job parameters
dbutils.widgets.text("bronzeTable", "delta_table", "Output Delta Table")
bronzeTable = dbutils.widgets.get("bronzeTable")

## Classify buildings into Commercial and Residential

In [ ]:
bronze = spark.read.table(bronzeTable)

# Explode the tags column to create key-value pairs
exploded_tags = (
    bronze
    .withColumn("key", F.explode(F.map_keys("tags")))
    .withColumn("value", F.col("tags")[F.col("key")])
)

# Define filters for residential and commercial buildings
residential_values = ["house", "residential", "detached", "terrace", "semidetached_house"]
commercial_values = ["commercial", "office", "retail", "service"]

# Explode the tags column and categorize buildings
categorized_buildings = (
    bronze.withColumn("key", F.explode(F.map_keys("tags")))
    .withColumn("value", F.col("tags")[F.col("key")])
    .filter((F.col("key") == "building") & F.col("geometry").isNotNull())
    .withColumn(
        "category",
        F.when(F.col("value").isin(*residential_values), "Residential")
         .when(F.col("value").isin(*commercial_values), "Commercial")
         .otherwise(None)
    )
    .filter(F.col("category").isNotNull())
)

# Visualise Sample
display(categorized_buildings.limit(10))


## Add H3 index
This allows us to easily compare and aggregate the data at a fine spatial resolution.

In [ ]:
index_resolution = 8

# On DBR 18+ / Spark 4.1, ST_* functions require the native GEOMETRY type while
# h3_* functions still require STRING (WKT) — they can't both consume the same
# column. We sidestep the conflict by using the bronze table's `type` column
# (osmium's element type: 'n' = node, 'w' = way, 'a' = area) to drive the
# branching and pass the raw WKT string to the h3_ functions.
categorized_buildings_indexed = categorized_buildings.withColumn(
    "h3_indices",
    F.expr(f"""
        CASE
            WHEN type = 'n' THEN ARRAY(h3_pointash3string(geometry, {index_resolution}))
            WHEN type IN ('w', 'a') THEN h3_coverash3string(geometry, {index_resolution})
            ELSE ARRAY()
        END
    """)
)


# Flatten H3 indices for aggregation
h3_flattened = categorized_buildings_indexed.withColumn("h3_index", F.explode("h3_indices"))

# Visualise Sample
display(h3_flattened.limit(10))


In [ ]:
# Calculate density (count) for residential and commercial buildings per H3 index
density = (
    h3_flattened
    .groupBy("h3_index", "category").count()
    .groupBy("h3_index")
    .pivot("category", ["Residential", "Commercial"])
    .sum("count")
    .fillna(0)
    .withColumnRenamed("Residential", "residential_count")
    .withColumnRenamed("Commercial", "commercial_count")
)

In [ ]:
# Add dominance classification
density = density.withColumn(
    "dominance",
    F.when(F.col("residential_count") > F.col("commercial_count") * 2, "Residential Dominant")
     .when(F.col("commercial_count") > F.col("residential_count") * 2, "Commercial Dominant")
     .otherwise("Balanced")
)

In [ ]:
# Identify densest areas
densest_residential = density.orderBy(F.desc("residential_count")).limit(10)
densest_commercial = density.orderBy(F.desc("commercial_count")).limit(10)


## Visualise

In [ ]:
import geopandas as gpd
import folium
from pyspark.databricks.sql import functions as dbf
from shapely import wkt

In [ ]:
# Process densest_commercial
pdf_commercial = densest_commercial.withColumn("geom", dbf.h3_boundaryaswkt("h3_index")).toPandas()
pdf_commercial["geom"] = pdf_commercial["geom"].apply(wkt.loads)
gdf_commercial = gpd.GeoDataFrame(pdf_commercial, geometry="geom", crs="EPSG:4326")

# Process densest_residential
pdf_residential = densest_residential.withColumn("geom", dbf.h3_boundaryaswkt("h3_index")).toPandas()
pdf_residential["geom"] = pdf_residential["geom"].apply(wkt.loads)
gdf_residential = gpd.GeoDataFrame(pdf_residential, geometry="geom", crs="EPSG:4326")

# Calculate combined bounds for auto-centering
bounds_commercial = gdf_commercial.total_bounds
bounds_residential = gdf_residential.total_bounds
min_lat = min(bounds_commercial[1], bounds_residential[1])
min_lon = min(bounds_commercial[0], bounds_residential[0])
max_lat = max(bounds_commercial[3], bounds_residential[3])
max_lon = max(bounds_commercial[2], bounds_residential[2])

# Create Folium Map
m = folium.Map()
m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])


# Add densest_commercial to the map (blue color)
for _, row in gdf_commercial.iterrows():
    folium.GeoJson(
        data=row["geom"].__geo_interface__,
        style_function=lambda x: {"color": "blue", "weight": 1},
        tooltip=(
            f"<b>Type:</b> Commercial<br>"
            f"<b>Residential Count:</b> {row['residential_count']}<br>"
            f"<b>Commercial Count:</b> {row['commercial_count']}"
        ),
    ).add_to(m)

# Add densest_residential to the map (red color)
for _, row in gdf_residential.iterrows():
    folium.GeoJson(
        data=row["geom"].__geo_interface__,
        style_function=lambda x: {"color": "red", "weight": 1},
        tooltip=(
            f"<b>Type:</b> Residential<br>"
            f"<b>Residential Count:</b> {row['residential_count']}<br>"
            f"<b>Commercial Count:</b> {row['commercial_count']}"
        ),
    ).add_to(m)

# Display the map in Databricks
html = m._repr_html_()
displayHTML(html)